# Data Inspection & Vector Database Analysis

This notebook explores the loaded Wikipedia data, embeddings, and vector database statistics.

**Goal**: Understand the data ingestion pipeline and vector database contents

In [ ]:
import sys
sys.path.insert(0, '../src')

from ingestion import DataIngestionService
from retrieval import RetrievalService
import json
from pathlib import Path

print("Modules imported successfully")

## 1. Initialize Services

In [ ]:
# Initialize the data ingestion service
ingestion_service = DataIngestionService()
retrieval_service = RetrievalService()

print(f"Services initialized")
print(f"Vector store path: {ingestion_service.persist_dir}")

## 2. Load Data (if not already loaded)

In [ ]:
# Check if data is already loaded
doc_count = ingestion_service.get_document_count()
print(f"Documents in index: {doc_count}")

if doc_count == 0:
    print("\nLoading Wikipedia data...")
    result = await ingestion_service.ingest_wikipedia_data(max_documents=100)
    print(f"Ingestion result: {result}")
    doc_count = ingestion_service.get_document_count()
    print(f"Documents after ingestion: {doc_count}")
else:
    print("Data already loaded. Skipping ingestion.")

## 3. Vector Database Statistics

In [ ]:
# Get index statistics
try:
    stats = retrieval_service.get_index_stats()
    print("Vector Database Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")
except Exception as e:
    print(f"Could not get stats: {e}")

## 4. Examine Embeddings

In [ ]:
# Generate an embedding for a sample query
sample_query = "What is machine learning?"

try:
    embedding = retrieval_service.get_embedding(sample_query)
    print(f"Query: {sample_query}")
    print(f"Embedding dimension: {len(embedding)}")
    print(f"First 10 values: {embedding[:10]}")
    print(f"Embedding type: {type(embedding)}")
except Exception as e:
    print(f"Error generating embedding: {e}")

## 5. Metadata Summary

In [ ]:
# Load and display metadata
metadata = ingestion_service.get_metadata()
if metadata:
    print("Ingestion Metadata:")
    print(json.dumps(metadata, indent=2))
else:
    print("No metadata file found")

## 6. Sample Documents from Index

In [ ]:
# Get the index and examine a few documents
try:
    index = ingestion_service.get_index()
    docstore = index.docstore.docs
    
    # Show first 3 documents
    print(f"Total documents in store: {len(docstore)}\n")
    
    for i, (doc_id, doc) in enumerate(list(docstore.items())[:3]):
        print(f"\n--- Document {i+1} ---")
        print(f"ID: {doc_id}")
        print(f"Text length: {len(doc.text)} chars")
        print(f"Text preview: {doc.text[:200]}...")
        if doc.metadata:
            print(f"Metadata: {doc.metadata}")
except Exception as e:
    print(f"Error examining documents: {e}")

## 7. Summary Statistics

In [ ]:
# Calculate summary statistics
try:
    index = ingestion_service.get_index()
    docstore = index.docstore.docs
    
    doc_lengths = [len(doc.text) for doc in docstore.values()]
    
    import statistics
    
    print("Document Statistics:")
    print(f"  Total documents: {len(doc_lengths)}")
    print(f"  Average document length: {statistics.mean(doc_lengths):.0f} characters")
    print(f"  Min length: {min(doc_lengths)} characters")
    print(f"  Max length: {max(doc_lengths)} characters")
    print(f"  Median length: {statistics.median(doc_lengths):.0f} characters")
    print(f"  Std deviation: {statistics.stdev(doc_lengths):.0f} characters")
    print(f"\nTotal text volume: {sum(doc_lengths):,} characters")
except Exception as e:
    print(f"Error calculating statistics: {e}")